In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/horror_articles.csv")
df.head()

,title,url,rating,subtitle,author,published,claim
0,Did American TV Viewers Hear the Devil's Voice...,https://www.snopes.com/fact-check/devils-voice...,false,This appears to be an urban legend of recent o...,David Emery,2023-03-25,"""On August 29th, 1968, all the televisions in ..."
1,Does This Video Show a 'Jersey Devil' Perched ...,https://www.snopes.com/fact-check/video-jersey...,false,The famous cryptid is part of New Jersey folkl...,Bethania Palma,2022-12-11,"A video shows a real ""Jersey Devil"" cryptid pe..."
2,Did a Power Surge in a Plugged-In Air Fryer Ca...,https://www.snopes.com/fact-check/air-fryer-fi...,true,Readers asked us to verify the contents of a F...,Jordan Liles,2022-11-30,A plugged-in air fryer underwent a power surge...
3,"No, This Is Not a 'Human Skin Jacket' from Bal...",https://www.snopes.com/fact-check/balenciaga-h...,false,It only takes a few seconds to overlay somebod...,David Emery,2022-12-03,A viral photo emblazoned with fashion house Ba...
4,Are Kids 'Snorting' Smarties and Putting Thems...,https://www.snopes.com/fact-check/snorting-sma...,legend,"Citing a doctor, a 2009 warning to parents fro...",David Mikkelson,2014-01-22,Schoolkids are “snorting” and “smoking” Smarti...


In [3]:
df["year"] = pd.to_datetime(df["published"]).dt.year

# fitur numerik sederhana dari teks
df["claim_len"]    = df["claim"].str.len()
df["subtitle_len"] = df["subtitle"].str.len()
df["title_len"]    = df["title"].str.len()
df["claim_words"]  = df["claim"].str.split().str.len()

# gabungkan penulis langka supaya one-hot tidak terlalu jarang
top = df["author"].value_counts()
df["author"] = df["author"].where(df["author"].isin(top[top >= 5].index), "other")

X = df.drop(columns=["year", "published", "title", "url", "subtitle", "claim"])
y = df["year"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
preprocessor = ColumnTransformer([
    ("numeric",  num_pipe(), ["claim_len", "subtitle_len", "title_len", "claim_words"]),
    ("categoric", cat_pipe(encoder="onehot"), ["author", "rating"]),
])

In [5]:
from xgboost import XGBRegressor
pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', XGBRegressor(n_jobs=-1, random_state=42,  eval_metric='mlogloss'))
])

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(pipeline, rsp.xgb_params, cv=3, scoring='r2', n_iter=25, n_jobs=-1, verbose=1, random_state=42)
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))

Fitting 3 folds for each of 25 candidates, totalling 75 fits
{'algo__colsample_bytree': np.float64(0.8347886858974456), 'algo__gamma': 1, 'algo__learning_rate': np.float64(0.020018663891407017), 'algo__max_depth': 6, 'algo__n_estimators': 179, 'algo__reg_alpha': np.float64(0.0014557961490251987), 'algo__reg_lambda': np.float64(0.23097861492693586), 'algo__subsample': np.float64(0.6387821809211413)}
0.9133086204528809 0.43582671880722046 0.5564144253730774


In [7]:
import pandas as pd

best = model.best_estimator_
names = best.named_steps["prep"].get_feature_names_out()
imp = pd.Series(best.named_steps["algo"].feature_importances_, index=names)
print(imp.sort_values(ascending=False).head(10))

categoric__author_other                0.229304
categoric__author_Bethania Palma       0.106461
categoric__author_Kim LaCapria         0.091110
categoric__author_Barbara Mikkelson    0.074969
categoric__author_David Mikkelson      0.062370
numeric__title_len                     0.049852
categoric__author_Snopes Staff         0.048819
categoric__rating_mostly false         0.045160
numeric__claim_words                   0.041562
categoric__rating_mixture              0.036140
dtype: float32
